# Project 10 — Varying Intercepts (Hierarchical Logistic)

**Scenario.** A binary binding assay is run across many protein *families*. Each family has its own baseline binding propensity (a group-varying intercept on the log-odds scale), and a global covariate $x$ shifts the log-odds for every observation.

**New skill:** group-level structure inside a **GLM** — a hierarchical logistic regression with varying intercepts. **Key pitfall:** with few groups it is easy to **pool too aggressively**, collapsing real family-to-family differences toward a single intercept. The amount of pooling is governed by the between-family SD $\tau$ and its prior.

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
az.style.use('arviz-darkgrid')
RNG = 20240601

## Step 1 — Problem & data-generating story

Families are **exchangeable**: their intercepts $\alpha_g$ are draws from a common population $\text{Normal}(\mu, \tau)$.

$$\alpha_g \sim \text{Normal}(\mu, \tau), \quad \text{logit}(p_i) = \alpha_{g[i]} + \beta x_i, \quad y_i \sim \text{Bernoulli}(p_i).$$

**Assumptions:** (a) families exchangeable, (b) Normal population of intercepts, (c) a single global slope $\beta$ shared across families, (d) Bernoulli outcomes. Truth: $\mu=-0.3,\ \tau=0.9,\ \beta=1.2$.

In [ ]:
from data.generate_data import generate
data = generate()
y, x, group, G = data['y'], data['x'], data['group'], data['G']
print(f"{G} families x {data['n_per']} obs = {len(y)} observations")
rates = np.array([y[group==g].mean() for g in range(G)])
print('per-family empirical binding rates:', np.round(rates, 2))

## Step 2 — Model specification with justified priors

Hyperpriors: $\mu \sim \text{Normal}(0, 1.5)$ and $\beta \sim \text{Normal}(0, 1.5)$ are weakly informative on the **log-odds** scale (an SD of 1.5 already spans binding probabilities from ~0.05 to ~0.95). $\tau \sim \text{HalfNormal}(1)$ is a weakly-informative positive scale prior; it must not be so tight that it forces all families to share one intercept (over-pooling), nor so vague it lets a 10-family dataset invent spurious spread.

**Non-centered** as in Project 09: $\alpha_g = \mu + \tau z_g$, $z_g \sim \text{Normal}(0,1)$ — the funnel hazard is identical in a GLM.

In [ ]:
from model import build_model, fit
model = build_model(data, parameterization='noncentered')
model

## Step 3 — Prior predictive checks

On the log-odds scale a careless prior can push every implied probability to 0 or 1. We simulate from the prior and check the implied binding rates spread sensibly across $[0,1]$ rather than piling at the extremes.

In [ ]:
with model:
    prior = pm.sample_prior_predictive(draws=400, random_seed=RNG)
pp = prior.prior_predictive['y'].values.reshape(-1, len(y)).mean(axis=1)
fig, ax = plt.subplots(figsize=(6,3.5))
ax.hist(pp, bins=30, color='#55A868', edgecolor='white')
ax.set(xlabel='dataset binding rate implied by prior', ylabel='count',
       title='Prior predictive — sensible spread, not piled at 0/1')
plt.tight_layout()

## Step 4 — Inference (NUTS, non-centered)

`draws=800, tune=1000, chains=4, target_accept=0.9`. Four chains for $\hat R$, a mild `target_accept` bump for the hierarchical geometry.

In [ ]:
idata = fit(data, parameterization='noncentered', draws=800, tune=1000,
            chains=4, target_accept=0.9, seed=101)

## Step 5 — Computational diagnostics

Check $\hat R$, ESS, **divergences = 0**, and the **energy plot**. The energy plot is the hierarchical canary: a marginal/transition mismatch signals the funnel even if $\hat R$ looks fine.

In [ ]:
print(az.summary(idata, var_names=['mu','tau','beta']))
print('divergences:', int(idata.sample_stats['diverging'].sum()))

In [ ]:
az.plot_energy(idata); plt.tight_layout()

In [ ]:
az.plot_trace(idata, var_names=['mu','tau','beta']); plt.tight_layout()

## Step 6 — Posterior predictive checks

Compare observed per-family binding counts to the posterior-predictive distribution. A good fit reproduces the spread of family rates.

In [ ]:
az.plot_ppc(idata, num_pp_samples=100); plt.tight_layout()

## Step 7 — Varying intercepts & shrinkage

Plot each family's no-pooling empirical log-odds against its partial-pooling posterior intercept $\alpha_g$. Families are pulled toward the population mean $\hat\mu$, and the noisiest (most extreme) are pulled most. This is the GLM version of Project 09's shrinkage. **The pitfall** is letting $\tau$ (via its prior) shrink so hard that all $\alpha_g$ collapse to $\hat\mu$ — see the broken notebook.

In [ ]:
eps = 0.5
emp_logodds = np.log((rates*data['n_per']+eps)/((1-rates)*data['n_per']+eps))
alpha_post = idata.posterior['alpha'].mean(dim=('chain','draw')).values
mu_hat = float(idata.posterior['mu'].mean())
fig, ax = plt.subplots(figsize=(6,4))
for g in range(G):
    ax.plot([0,1], [emp_logodds[g], alpha_post[g]], color='grey', alpha=0.6)
ax.scatter(np.zeros(G), emp_logodds, color='#C44E52', label='no pooling')
ax.scatter(np.ones(G), alpha_post, color='#4C72B0', label='partial pooling')
ax.axhline(mu_hat, color='k', ls='--', lw=1, label='mu_hat')
ax.set(xticks=[0,1], xticklabels=['no pool','partial'],
       ylabel='family intercept (log-odds)', title='Shrinkage of intercepts')
ax.legend(); plt.tight_layout()

## Step 8 — Decision & communication

Recover the population parameters and the global slope, then verify against known truth.

In [ ]:
from shared.bayes_utils import check_recovery
for res in check_recovery(idata, data['truth']):
    print(res)

**Conclusion (for a collaborator).** Higher physicochemical score $x$ raises binding odds ($\beta>0$, robustly). Families do differ in baseline propensity ($\tau\approx0.9$), but with only ~12 assays each, report the **shrunken** per-family intercepts, not the raw rates. See `summary_onepager.md`.